#### Define date parameters

In [ ]:
start_date = "2026-09-11"
end_date = "2026-09-17"

StatementMeta(, 8aac9347-e70a-4e80-970c-a2320f9c7302, 3, Finished, Available, Finished, False)

#### Import PySpark functions

In [ ]:
from pyspark.sql.functions import col, lit, to_timestamp

StatementMeta(, 8aac9347-e70a-4e80-970c-a2320f9c7302, 4, Finished, Available, Finished, False)

#### Read the Bronze JSON

In [ ]:
file_path = f"Files/{start_date}_earthquake_data.json"
df = (
spark.read
.option("multiLine", "true")
.json(file_path)
)
display(df)

StatementMeta(, 8aac9347-e70a-4e80-970c-a2320f9c7302, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d2fdc4cb-7b6c-45cc-af0f-1f95ca562840)

#### Extract useful fields

In [ ]:
df_silver = df.select(
col("id").alias("id"),
col("geometry.coordinates").getItem(0).alias("longitude"),
col("geometry.coordinates").getItem(1).alias("latitude"),
col("geometry.coordinates").getItem(2).alias("depth"),
col("properties.title").alias("title"),
col("properties.place").alias("place"),
col("properties.mag").alias("magnitude"),
col("properties.magType").alias("magnitude_type"),
col("properties.sig").alias("significance"),
col("properties.time").alias("time"),
col("properties.updated").alias("updated")
)
display(df_silver)

StatementMeta(, 8aac9347-e70a-4e80-970c-a2320f9c7302, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 10a2fcbd-d179-4f73-8181-be5a139d7bf1)

#### Convert Unix milliseconds to timestamps

In [ ]:
spark.conf.set("spark.sql.session.timeZone", "UTC")
df_silver = (
df_silver
.withColumn("time", (col("time") / 1000).cast("timestamp"))
.withColumn("updated", (col("updated") / 1000).cast("timestamp"))
)
display(df_silver)

StatementMeta(, 8aac9347-e70a-4e80-970c-a2320f9c7302, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 164c5124-d514-4c54-a6ae-a9d77054d1e4)

#### Restrict records to the selected date window

In [ ]:
df_silver = df_silver.filter(
(col("time") >= to_timestamp(lit(start_date))) &
(col("time") < to_timestamp(lit(end_date)))
)

StatementMeta(, 8aac9347-e70a-4e80-970c-a2320f9c7302, 8, Finished, Available, Finished, False)

#### Save the Silver Delta table

In [ ]:
(
df_silver.write
.format("delta")
.mode("append")
.saveAsTable("earthquake_events_silver")
)

StatementMeta(, 8aac9347-e70a-4e80-970c-a2320f9c7302, 9, Finished, Available, Finished, False)